## 5. Koşullu Akışlar (SequentialChain / ConditionalChain)
* Amaç: LLM’lerin çok aşamalı işlemleri nasıl zincirlediğini göstermek.
* Senaryo: Kullanıcıdan konu alınır → özet çıkarılır → başlık üretilir → tweet formatında yazılır.
* Kazandırılan: Çok adımlı işlem mantığı.
* Kullanılan Bileşenler: SimpleSequentialChain, LLMChain

In [ ]:
# ============================================================
# 1) KURULUM
# ============================================================
!pip install -q "langchain>=0.2" "langchain-community>=0.2" \
               transformers accelerate sentencepiece

In [ ]:

# ============================================================
# 2) İÇE AKTARIMLAR
# ============================================================
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_community.llms import HuggingFacePipeline
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain

import textwrap

# ============================================================
# 3) LLM: HF ÜZERİNDEN AÇIK KAYNAK MODEL
# ============================================================
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"  # Alternatif: "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID)

gen_pipe = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=220,
    do_sample=True,
    temperature=0.2,      # eğitimde tutarlı çıktılar için düşük sıcaklık
    top_p=0.9,
    repetition_penalty=1.05,
)

llm = HuggingFacePipeline(pipeline=gen_pipe)

# ============================================================
# 4) PROMPTLAR ve ZİNCİRLER
# ============================================================
# 4.1 ÖZET
summary_prompt = PromptTemplate(
    input_variables=["topic"],
    template=textwrap.dedent("""\
        Aşağıdaki konuyu 3-5 cümlede, açık ve akademik Türkçe ile özetle.
        Gereksiz tekrar ve retorikten kaçın; somut kavramları vurgula.

        Konu: {topic}

        Özet:
    """)
)
summary_chain = LLMChain(llm=llm, prompt=summary_prompt, output_key="summary")

# 4.2 BAŞLIK
title_prompt = PromptTemplate(
    input_variables=["summary"],
    template=textwrap.dedent("""\
        Aşağıdaki özeti temel alarak, en fazla 60 karakterlik,
        teknik fakat çekici bir Türkçe başlık üret. Nokta kullanma.

        Özet:
        {summary}

        Başlık:
    """)
)
title_chain = LLMChain(llm=llm, prompt=title_prompt, output_key="title")

# 4.3 TWEET (<= 240 karakter)
tweet_prompt = PromptTemplate(
    input_variables=["title", "summary"],
    template=textwrap.dedent("""\
        Aşağıdaki başlık ve özete dayanarak 240 karakteri geçmeyen,
        bilgilendirici bir Türkçe tweet yaz. 1-2 uygun hashtag ekle.

        Başlık: {title}
        Özet: {summary}

        Tweet:
    """)
)
tweet_chain = LLMChain(llm=llm, prompt=tweet_prompt, output_key="tweet")

# 4.4 (OPSİYONEL / KOŞULLU) TL;DR
tldr_prompt = PromptTemplate(
    input_variables=["summary"],
    template=textwrap.dedent("""\
        Aşağıdaki özeti tek cümlede TL;DR biçiminde yoğunlaştır.
        Öz cümleyi üret; 30 kelimeyi geçmesin.

        Özet:
        {summary}

        TL;DR:
    """)
)
tldr_chain = LLMChain(llm=llm, prompt=tldr_prompt, output_key="tldr")

# ============================================================
# 5) KOŞULLU AKIŞ YÜRÜTÜCÜ
# ============================================================
def run_pipeline(topic: str, tldr_threshold: int = 600):
    # Adım 1: Özet
    summary = summary_chain.run(topic).strip()

    # Adım 2: Başlık
    title = title_chain.run(summary).strip()

    # Adım 3: Tweet
    tweet = tweet_chain.run(title=title, summary=summary).strip()

    # Adım 4 (KOŞULLU): Özet çok uzunsa TL;DR üret
    # Burada metin uzunluğunu karakter bazlı kontrol ediyoruz (eğitsel amaçlı basit kural).
    if len(summary) > tldr_threshold:
        tldr = tldr_chain.run(summary=summary).strip()
    else:
        tldr = "Özet yeterince kısa; TL;DR gerekli görülmedi."

    return {
        "summary": summary,
        "title": title,
        "tweet": tweet,
        "tldr": tldr,
        "summary_len": len(summary)
    }

# ============================================================
# 6) DEMO
# ============================================================
topic = """\
Büyük dil modellerinde (LLM) akıl yürütme kabiliyetini artırmak için kullanılan
Chain-of-Thought (CoT), Self-Consistency ve Toolformer benzeri yöntemlerin
karşılaştırılması; parametre ölçeği, veri kalitesi ve yönlendirme (prompting)
stratejilerinin sonuç doğruluğuna etkisi.
"""

result = run_pipeline(topic, tldr_threshold=550)

print("\n=== ÖZET ({} karakter) ===\n".format(result["summary_len"]))
print(result["summary"])

print("\n=== BAŞLIK ===\n")
print(result["title"])

print("\n=== TWEET (<=240) ===\n")
print(result["tweet"])

print("\n=== TL;DR (Koşullu) ===\n")
print(result["tldr"])
